# 02 — Data preparation

**Machine Learning · M.EIC001 · Group G51**

This notebook covers the **data preparation** phase of CRISP-DM. It builds one table with one row per team-season, holding the targets of that season and features computed only from earlier seasons.

**Main rule:** we predict the day before the first game of season `t`, so features can only use data up to season `t-1`.

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

RANDOM_STATE = 42
DATA_DIR = Path("hockey_development_data") / "hockey_development_data"
FIRST_TARGET_YEAR = 1974  # first season with confID

## 1. Load and scope

We keep the NHL only. Earlier NHL seasons stay in the tables for now because they can feed history features, but only seasons from 1974 become target rows.

Free-text columns are dropped. `Scoring` rows without `GP` are also dropped: in 238 of the 249 NHL cases the player has no other row with stats in that year.

In [2]:
def load_nhl(name):
    df = pd.read_csv(DATA_DIR / f"{name}.csv", low_memory=False)
    df = df.query("lgID == 'NHL'").copy()
    return df.drop(columns=["note", "notes", "nameNote"], errors="ignore")

teams = load_nhl("Teams")
coaches = load_nhl("Coaches")
scoring = load_nhl("Scoring").dropna(subset=["GP"])
goalies = load_nhl("Goalies").dropna(subset=["GP"])
master = pd.read_csv(DATA_DIR / "Master.csv", low_memory=False)

pd.Series({"Teams": len(teams), "Coaches": len(coaches), "Scoring": len(scoring),
           "Goalies": len(goalies), "Master": len(master)}, name="rows")

Teams       1265
Coaches     1522
Scoring    39852
Goalies     3660
Master      7469
Name: rows, dtype: int64

## 2. Targets

**Task (a).** `Teams.rank` is the **division** rank, so the conference rank is computed here. `pts_pct` is `Pts / (2 * G)`: `Pts = 2W + T + OTL` holds in every NHL row, and since 2005 shootout losses are already inside `OTL`. Teams level on points are separated by wins.

**Task (b).** A team changes coach if more than one coach has games in that season. `MIN_STINT_GAMES` controls which spells count. There are very short spells (14 of 1 to 3 games since 1974), so this value is still to be decided with the group.

In [3]:
MIN_STINT_GAMES = 1

teams["pts_pct"] = teams["Pts"] / (2 * teams["G"])

ranked = (teams[teams["confID"].notna()]
          .sort_values(["year", "confID", "pts_pct", "W"], ascending=[True, True, False, False]))
teams["conf_rank"] = ranked.groupby(["year", "confID"]).cumcount() + 1

n_coaches = (coaches[coaches["g"] >= MIN_STINT_GAMES]
             .groupby(["year", "tmID"])["coachID"].nunique()
             .rename("n_coaches").reset_index())
teams = teams.merge(n_coaches, on=["year", "tmID"], how="left")
teams["coach_changed"] = (teams["n_coaches"] > 1).astype(int)

targets_1974 = teams[teams["year"] >= FIRST_TARGET_YEAR]
pd.Series({"team-seasons": len(targets_1974),
           "coach change rate": targets_1974["coach_changed"].mean(),
           "rows without conf_rank": targets_1974["conf_rank"].isna().sum()}, name="targets")

team-seasons              842.000
coach change rate           0.198
rows without conf_rank      0.000
Name: targets, dtype: float64

## 3. Team features

These are computed per season first and only moved to the next season in section 4.

Each one is converted to a z-score inside its season, so that values are relative to the league of that year and comparable across eras.

In [4]:
teams["goal_diff_g"] = (teams["GF"] - teams["GA"]) / teams["G"]
teams["pim_g"] = teams["PIM"] / teams["G"]
teams["pp_pct"] = teams["PPG"] / teams["PPC"]
teams["pk_pct"] = 1 - teams["PKG"] / teams["PKC"]

SEASON_STATS = ["pts_pct", "goal_diff_g", "pim_g", "pp_pct", "pk_pct"]

def season_zscore(s):
    return (s - s.mean()) / s.std()

for col in SEASON_STATS:
    teams[f"{col}_z"] = teams.groupby("year")[col].transform(season_zscore)

teams[[f"{c}_z" for c in SEASON_STATS]].describe().loc[["count", "mean", "std"]]

,pts_pct_z,goal_diff_g_z,pim_g_z,pp_pct_z,pk_pct_z
count,1.265e+03,1.265e+03,1.265e+03,9.620e+02,9.620e+02
mean,3.704e-17,7.021e-19,-2.826e-17,-2.954e-17,3.370e-16
std,9.633e-01,9.633e-01,9.633e-01,9.763e-01,9.763e-01


## 4. Franchise history

`tmID` changes when a team moves or is renamed, so history follows `franchID`. `shift` takes the previous season the franchise played, so the season before 2005 is 2003 (2004 was the lockout). `year - 1` would get this wrong.

- `_lag1`: value in the previous season
- `_avg3`: mean of the previous three seasons, shifted so season `t` is never included

Teams filtered out of the NHL have no earlier row here, so the four franchises that joined from the WHA in 1979 are treated as new teams.

In [5]:
teams = teams.sort_values(["franchID", "year"]).reset_index(drop=True)
by_franchise = teams.groupby("franchID")

history = {}
for col in [f"{c}_z" for c in SEASON_STATS]:
    history[f"{col}_lag1"] = by_franchise[col].shift(1)
    history[f"{col}_avg3"] = by_franchise[col].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean())
history["coach_changed_lag1"] = by_franchise["coach_changed"].shift(1)

teams = pd.concat([teams, pd.DataFrame(history)], axis=1)
teams["is_new_team"] = teams["pts_pct_z_lag1"].isna().astype(int)

FEATURES = list(history) + ["is_new_team"]
print(len(FEATURES), "features")

12 features


## 5. Build the panel

One row per team-season from 1974. It keeps the keys, the two targets and the features. Raw columns of season `t` are left out, which is what prevents leakage.

Missing values stay as they are: imputation and scaling must be fitted inside each training fold, in a `Pipeline`, during modelling.

In [6]:
KEYS = ["year", "franchID", "tmID", "confID", "divID"]
TARGETS = ["pts_pct", "conf_rank", "coach_changed"]

panel = (teams.loc[teams["year"] >= FIRST_TARGET_YEAR, KEYS + TARGETS + FEATURES]
              .sort_values(["year", "confID", "conf_rank"])
              .reset_index(drop=True))

# checks on the table
assert not panel.duplicated(["year", "tmID"]).any()
assert panel["conf_rank"].notna().all()
assert all(c.endswith(("_lag1", "_avg3")) or c == "is_new_team" for c in FEATURES)

# leakage check: a lagged feature must match the franchise's previous season
det = teams.query("franchID == 'DET'").set_index("year")
assert np.isclose(det.loc[2005, "pts_pct_z_lag1"], det.loc[2003, "pts_pct_z"])

print(panel.shape, "| seasons:", panel["year"].nunique(), "| new teams:", panel["is_new_team"].sum())
panel.head()

(842, 20) | seasons: 35 | new teams: 15


,year,franchID,tmID,confID,divID,pts_pct,conf_rank,coach_changed,pts_pct_z_lag1,pts_pct_z_avg3,goal_diff_g_z_lag1,goal_diff_g_z_avg3,pim_g_z_lag1,pim_g_z_avg3,pp_pct_z_lag1,pp_pct_z_avg3,pk_pct_z_lag1,pk_pct_z_avg3,coach_changed_lag1,is_new_team
0,1974,PHI,PHI,CC,PT,0.706,1.0,0,1.553,0.454,1.430,0.489,3.216,2.781,0.627,0.900,2.110,0.999,0.0,0
1,1974,NYR,NYR,CC,PT,0.550,2.0,0,0.731,1.018,0.643,1.083,-0.687,-0.354,2.158,1.281,0.148,0.745,1.0,0
2,1974,NYI,NYI,CC,PT,0.550,3.0,0,-1.005,-1.547,-0.853,-1.489,0.495,0.112,-1.104,-1.384,-1.271,-1.439,0.0,0
3,1974,VAN,VAN,CC,SM,0.537,4.0,0,-0.868,-1.064,-0.945,-1.124,-0.001,0.295,0.092,-0.199,-1.808,-1.431,1.0,0
4,1974,STL,STL,CC,SM,0.525,5.0,1,-0.639,-0.393,-0.551,-0.416,0.785,1.027,-1.049,-1.037,0.403,-0.264,1.0,0


## 6. Baseline

Each new group of features should be compared with this. The baseline orders every conference by the previous season, and new teams get the mean. Each conference-season gets its own Spearman correlation, then these are averaged.

In [7]:
def rank_scores(df, score_col):
    pred = df.groupby(["year", "confID"])[score_col].rank(ascending=False, method="first")
    spearman = [spearmanr(pred[g.index], g["conf_rank"])[0] for _, g in df.groupby(["year", "confID"])]
    return {"spearman": np.mean(spearman), "rank_mae": (pred - df["conf_rank"]).abs().mean()}

baseline = panel.assign(score=panel["pts_pct_z_lag1"].fillna(0))  # z-score mean is 0

pd.DataFrame({
    "1995-2009": rank_scores(baseline[baseline["year"] >= 1995], "score"),
    "2005-2009": rank_scores(baseline[baseline["year"] >= 2005], "score"),
}).T

,spearman,rank_mae
1995-2009,0.549,3.027
2005-2009,0.501,3.307


## 7. Next steps

- [ ] Decide `MIN_STINT_GAMES` and the rank tie-break after talking to the professor
- [ ] Roster features from `Scoring` and `Master`: average age, points of the best players, recent awards. Aggregate by player and team first, because traded players have several rows
- [ ] Goalie features from `Goalies`: save percentage and goals against of the starter. `SA` only starts in 1982, so it needs a model-based imputation
- [ ] Coach features from `Coaches`: seasons in the job, career win percentage
- [ ] Form features from `TeamSplits` and `TeamsPost`: home and road records, late-season results, playoff round reached
- [ ] Outlier handling for linear models (`RobustScaler` or winsorizing)
- [ ] Feature selection and SMOTE inside the training folds